# Session 04: Sampling and Generation

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [04_outputs.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/04_outputs.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes; `Course completion:` comments identify
restored steps. Original classroom notes, including `##` comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-04/`. This repaired version awaits a new Colab run.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-04"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:
##Attention and hidden states (very briefly)

# Attention is how we get or build context from the static embeddings that any model stores
# Attention is run multiple times in a model; GPT 2 --> 12x
# Affter attention, we run outputs through a feedforward network (Rosenblatt, Jed Dobson)
# ^ All of the above produces a hidden state
# THen, finally, we pass all of the above (x12) througha linear layer, or "unembedding layer"

# Model outputs.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)
torch.manual_seed(5167)
torch.set_grad_enabled(False)
model.eval()



In [ ]:
tokenizer.pad_token = tokenizer.eos_token
text = "The work of art in the age of mechanical"
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model(**inputs)

In [ ]:
logits = outputs.logits
print("Shape of logits:", list(logits.shape)) # BAtch, num token, ....

In [ ]:
assert inputs["input_ids"].size(1) == logits.size(1), "Unmatched size"
assert model.config.vocab_size == logits.size(2), "Unmatched size"

In [ ]:
logits = logits[0, -1,  :]

In [ ]:
# logits: raw, unnormalizzed scores from the model, which indicate what the model thinks
# should be the next tokens.


In [ ]:
# Softmax
logits

In [ ]:
torch.exp(torch.tensor([8])) #e^8

In [ ]:
z = torch.tensor([1.25, -0.3, 0.87])
z -= z.max()

exp = torch.exp(z)
sigma_z = exp/exp.sum()
sigma_z

In [ ]:
sigma_z.sum()

In [ ]:
# Repair: Report exponential overflow explicitly and plot its finite values.
A =  torch.arange(1000)
exp_A = torch.exp(A)
print("Float32 overflow entries:", torch.isinf(exp_A).sum().item())
finite = torch.isfinite(exp_A)
sns.lineplot(x=A[finite].numpy(), y=exp_A[finite].numpy())
save_figure("s04-original-013.png")

In [ ]:
# Convert the logits via Softmax to a probability distribution
probs = F.softmax(logits, dim=-1)
probs

In [ ]:
# whats the next most likely token?
index = torch.argmax(probs).item()
token = tokenizer.decode(index)
print("Most likely token:", repr(token)) # Input --> output 

In [ ]:
# LLMs are deterministic! Argmax is taking the highest value to the response.

print("Most likely token:", repr(token))

In [ ]:
# Course completion: Qualify the determinism note for the greedy example.
# Clarification: the fixed-input greedy choice is deterministic; multinomial sampling below is stochastic.


In [ ]:
idx = torch.argmax(probs).item()
print("Most likely token:", repr(tokenizer.decode(idx)))

# Top-25
k = 25
top_probs, top_indices = torch.topk(probs, k)
top_tokens = [repr(tokenizer.decode(i.item())) for i in top_indices]
top_df = pd.DataFrame({"token": top_tokens, "probability": top_probs.numpy()})
print(top_df)

# 
fig, ax = plt.subplots()
sns.barplot(top_df, x="token", y="probability", ax=ax)
ax.tick_params(axis='x', rotation=90)
ax.set_title("Top 25 Probable Next Tokens")
save_figure("s04-original-017.png")

In [ ]:
fig, ax = plt.subplots()
sns.histplot(
    probs.detach().cpu().numpy(),
    stat="density",
    element="step",
    log_scale=True,
    ax=ax
)
ax.set(title="Token Probability Distribution", xlabel="Probability (log)")
save_figure("s04-original-018.png")

In [ ]:
# sampling: make a decision w.r.t the next token.


# 1. greedy decoding.
def greedy_decoding(probs):
    """Perform greedy decoding.

    Parameters
    ----------
    probs : torch.Tensor, shape (vocab_size,)
        Token probabilities

    Returns
    -------
    int
        Selected token ID
    """
    idx = torch.argmax(probs)

    return idx.item()

In [ ]:
idx = greedy_decoding(probs)
token = tokenizer.decode(idx)
print(repr(text + token))




In [ ]:
for _ in range(5):
    idx = greedy_decoding(probs)
    token = tokenizer.decode(idx)
    print(repr(text + token))



In [ ]:

# 2. multinominal sampling: treates probilities as exactly that: probabiliies, such that a token with probabilty .6
# will be chosen 60% of the time, a token with prob 0.001 will be chosen.
#quite rarely.
def multinomial_sampling(probs):
    """Perform multinomial sampling.

    Parameters
    ----------
    probs : torch.Tensor, shape (vocab_size,)
        Token probabilities

    Returns
    -------
    int
        Sampled token ID
    """
    idx = torch.multinomial(probs, num_samples=1)

    return idx.item()

for _ in range(5):
    idx = multinomial_sampling(probs)
    token = tokenizer.decode(idx)
    print(repr(text + token))

In [ ]:
# 3. Top-k sampling: limits the candidate pool tot he top k most probable tokens.
## initialize a vector 0 prob and assign values from probs to indicies that correspond to the topk V.


def topk_filtering(probs, k=50):
    """Filter a probability distribution to only the top-k tokens.

    Parameters
    ----------
    probs : torch.Tensor, shape (vocab_size,)
        Token probabilities
    k : int
        Top token pool size

    Returns
    -------
    torch.Tensor, shape (vocab_size,)
        Filtered probabilities

    Raises
    ------
    ValueError
        If k is less than 0
    """
    if k <= 0:
        raise ValueError("k must be greater than 0")

    k = min(k, probs.numel())
    values, indices = torch.topk(probs, k=k)

    # Initialize vector of 0 probabilities and assign values from probs to 
    # indices that correspond to the top-k values
    filtered = torch.zeros_like(probs)
    filtered[indices] = probs[indices]

    # Normalize probabilities
    filtered = filtered / filtered.sum()

    return filtered



topk_probs = topk_filtering(probs)

for _ in range(5):
    idx = multinomial_sampling(topk_probs)
    token = tokenizer.decode(idx)
    print(repr(text + token))

In [ ]:
# Repair: A logarithmic histogram requires positive probabilities; filtered-out zeros are not candidates.
sns.histplot(topk_probs[topk_probs > 0].detach().cpu().numpy(), log_scale=True)
save_figure("s04-original-025.png")

In [ ]:
# Repair: Sample from the nucleus-filtered distribution in the top-p example.
# Repair: The top-p example reused top-k and contained an unreachable indented assignment.
# 4. Top-p sampling, or nucleus sampling: limits the candidate pool by the cumulative
# probability of tokens. Instead of setting a hard hnumber k, you dynamically set the
# site of your token pool based on the cumulative probability you want.
#
# Holtzmann (2019): how do you limit repetition in model outputs?


def topp_filtering(probs, p=0.9):
    """Filter a probability distribution using top-p/nucleus filtering.

    Parameters
    ----------
    probs : torch.Tensor, shape (vocab_size,)
        Token probabilities
    p : float
        Cumulative probability

    Returns
    -------
    torch.Tensor, shape (vocab_size,)
        Filtered probabilities

    Raises
    ------
    ValueError
        If p is less than or equal to 0, or greather than 1
    """
    if not 0 < p <= 1:
        raise ValueError("p must be in the interval (0, 1]")

    values, indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(values, dim=0)

    # Index of the first token where cumulative probability >= p. Add 1 to keep
    # tokens up to and including the cutoff
    p = torch.tensor(p, device=probs.device, dtype=probs.dtype)
    cutoff = torch.searchsorted(cumulative, p).item() + 1
    candidate_indices = indices[:cutoff]

    # Normalize probabilities
    filtered = torch.zeros_like(probs)
    filtered[candidate_indices] = probs[candidate_indices]

    filtered = filtered / filtered.sum()

    return filtered



    
topp_probs = topp_filtering(probs)

for _ in range(5):
    idx = multinomial_sampling(topp_probs)
    token = tokenizer.decode(idx)
    print(repr(text + token))

In [ ]:

# 5. Temperature scaling reshapes the underlying.

# logits / t
# t = temperature (0, inf)
# - t < 1: making the model outputs more deterministic, aka closer to greedy decoding
# - t = 1: no effect
# - t > 1: make model outputs more random
#
# Modifying the probability distribution by either sharpening (t < 1) or flattening it (t > 1)
#
# Temperature scheduling: during model trainig, modify temprature
#
# Derives from statistical physics, entropy, Shannon (And kolmogorov ;D)
#
### Higher temperature would leads to flattening the distribution of outputs. 

def temperature_scaling(logits, temperature=1.0):
    """Perform temperature scaling.

    Parameters
    ----------
    logits : torch.Tensor
        Output logits, shape (vocab_size,)
    temperature : float
        Scaling factor. Higher values make the distribution flatter; lower
        values make it sharper

    Returns
    -------
    torch.Tensor, shape (vocab_size,)
        Probabilities after temperature scaling

    Raises
    ------
    ValueError
        If temperature is below 0
    """
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")

    scaled = logits / temperature
    probs = F.softmax(scaled, dim=-1)

    return probs

In [ ]:
temp_probs = temperature_scaling(logits, temperature=1.5)

for _ in range(5):
    idx = multinomial_sampling(temp_probs)
    token = tokenizer.decode(idx)
    print(repr(text + token))

In [ ]:
def get_sampling_distribution(logits, greedy=False, temperature=1.0, k=None, p=None):
    """Get a sampling distribution."""
    probs = temperature_scaling(logits, temperature=temperature)

    if greedy:
        index = greedy_decoding(probs)
        probs = torch.zeros_like(probs)
        probs[index] = 1.0

        return probs

    if k is not None:
        probs = topk_filtering(probs, k=k)

    if p is not None: probs = topp_filtering(probs, p=p)

    return probs

distribution = get_sampling_distribution(logits, temperature=1.2, k=100, p=0.9)
for _ in range(5):
    index = multinomial_sampling(distribution)
    token = tokenizer.decode(index)
    print(repr(text + token))
    

In [ ]:
configs = {
    "greedy": dict(greedy=True),
    "multinomial": dict(),
    "temp=0.7": dict(temperature=0.7),
    "temp=1.3": dict(temperature=1.3),
    "top-k=50": dict(k=50),
    "top-p=0.9": dict(p=0.90),
    "top-k=50, top-p=0.9": dict(k=50, p=0.9),
    "temp=0.7, top-p=0.9": dict(temperature=0.7, p=0.9),
    "temp=1.3, top-p=0.9": dict(temperature=1.3, p=0.9),
}

for key, values in configs.items():
    print("Settings:", key)
    distribution = get_sampling_distribution(logits, **values)

    index = multinomial_sampling(distribution)
    token = tokenizer.decode(index)
    print(repr(text + token))

    print()


In [ ]:
# Repair: Stop the original generation loop on EOS.
new_tokens = 10
for name, config in configs.items():
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    seq = inputs["input_ids"]

    for _ in range(new_tokens):
        seq_outputs = model(input_ids=seq)

        seq_logits = seq_outputs.logits[0, -1, :]
        sampling_probs = get_sampling_distribution(seq_logits, **config)
        index = multinomial_sampling(sampling_probs)
        if index == tokenizer.eos_token_id:
            break

        ## Append the sampled token ID, shape (1, seq_len)
        next_token = torch.tensor([[index]], device=seq.device)
        seq = torch.cat([seq, next_token], dim=1)

    generated = tokenizer.decode(seq[0])

    print("Config:", name)
    print("    Text:", repr(generated), flush=True)

In [ ]:
# Course completion: Restore the temperature-distribution comparison.
temperatures = [0.5, 1.0, 1.5]
top_n = 500

rows = []
for temperature in temperatures:
    temp_probs = temperature_scaling(logits, temperature=temperature)
    sorted_probs, _ = torch.sort(temp_probs, descending=True)
    top_probs = sorted_probs[:top_n]

    for rank, prob in enumerate(top_probs, start=1):
        rows.append(
            {
                "temperature": "temp=" + str(temperature),
                "rank": rank,
                "prob": prob.item()
            }
        )


In [ ]:
# Course completion: Restore the temperature-distribution comparison.
temp_df = pd.DataFrame(rows)


In [ ]:
# Course completion: Restore the temperature-distribution comparison.
fig, ax = plt.subplots()
sns.lineplot(temp_df, x="rank", y="prob", hue="temperature", ax=ax)
ax.set_yscale("log")
ax.set(
    title="Token Probabilities by Temperature",
    xlabel="Token rank",
    ylabel="Probability",
)
ax.legend(title="Temperature")
save_figure(RESULTS_DIR / "figure-03.png")


In [ ]:
# Course completion: Restore the course EOS stopping comparison.
tokenizer.eos_token


In [ ]:
# Course completion: Restore the course EOS stopping comparison.
text = "News at 5."
max_new_tokens = 100
eos_token_id = tokenizer.eos_token_id


In [ ]:
# Course completion: Restore the course EOS stopping comparison.
eos_results = []
for name, config in configs.items():
    inputs = tokenizer(text, return_tensors="pt")
    seq = inputs["input_ids"]

    stopped_early = False

    for _ in range(max_new_tokens):
        seq_outputs = model(input_ids=seq)

        seq_logits = seq_outputs.logits[0, -1, :]
        sampling_probs = get_sampling_distribution(seq_logits, **config)
        idx = multinomial_sampling(sampling_probs)

        # Is the sampled token our EOS token? If so, stop
        if idx == eos_token_id:
            stopped_early = True
            break

        # Append the sampled token ID, shape (1, seq_len)
        next_token = torch.tensor([[idx]], device=seq.device)
        seq = torch.cat([seq, next_token], dim=1)

    generated = tokenizer.decode(seq[0])

    eos_results.append(
        {
            "config": name,
            "stopped_early": stopped_early,
            "text": generated,
        }
    )


In [ ]:
# Course completion: Restore the course EOS stopping comparison.
eos_df = pd.DataFrame(eos_results)
eos_df


In [ ]:
# Course completion: Restore distribution statistics inline.
def distribution_stats(probs):
    """Derive statistics from a probability distribution.

    Parameters
    ----------
    probs : torch.Tensor
        Token probabilities, shape (vocab_size,)

    Returns
    -------
    dict
        Statistics for support size, entropy, effective vocab, max probability,
        and expected rank
    """
    nonzero = probs > 0
    p_nonzero = probs[nonzero]

    entropy = -(p_nonzero * torch.log(p_nonzero)).sum()
    effective_vocab = torch.exp(entropy)

    sorted_probs, _ = torch.sort(probs, descending=True)
    ranks = torch.arange(1, probs.numel() + 1, device=probs.device)
    expected_rank = (sorted_probs * ranks).sum()

    return {
        "support_size": nonzero.sum().item(),
        "entropy": entropy.item(),
        "effective_vocab": effective_vocab.item(),
        "max_prob": probs.max().item(),
        "expected_rank": expected_rank.item(),
    }


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
dist_results = []
for name, kwargs in configs.items():
    probs = get_sampling_distribution(logits, **kwargs)
    stats = distribution_stats(probs)

    # Add a dictionary for each statistic
    for metric, value in stats.items():
        dist_results.append(
            {
                "method": name,
                "metric": metric,
                "value": value,
            }
        )


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
dist_df = pd.DataFrame(dist_results)

g = sns.catplot(
    data=dist_df,
    x="method",
    y="value",
    col="metric",
    col_wrap=2,
    kind="bar",
    sharey=False,
    height=FACET_HEIGHT,
    aspect=FACET_ASPECT,
)

titles = {
    "support_size": "Support size (log tokens)",
    "entropy": "Entropy (nats)",
    "effective_vocab": "Effective vocab (tokens)",
    "max_prob": "Max probability (0-1)",
    "expected_rank": "Expected rank (position)",
}

g.set_titles("{col_name}")
for ax in g.axes.flat:
    metric = ax.get_title()
    if metric == "support_size":
        ax.set_yscale("log")  # Use log scale since support_size can be huge

    ax.set_title(titles.get(metric, metric))

g.set_axis_labels("", "Value")
g.tick_params(axis="x", rotation=90)
g.fig.suptitle("Distribution Statistics for Different Sampling Methods", y=1.05)
save_figure(RESULTS_DIR / "figure-04.png")


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
text = "The work of art in the age of mechanical reproduction"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs, labels=inputs["input_ids"])


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
loss = outputs.loss
print("Cross-entropy:", loss.item())
print("Perplexity:", torch.exp(loss).item())


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
perp_results = []
for seq in eos_df["text"]:
    inputs = tokenizer(seq, return_tensors="pt")
    outputs = model(**inputs, labels=inputs["input_ids"])

    perp = torch.exp(outputs.loss).item()
    perp_results.append(perp)


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
eos_df["perplexity"] = perp_results


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
eos_df.sort_values("perplexity", ascending=False)


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
lowest_perp = eos_df.nsmallest(1, "perplexity")
print(lowest_perp["text"].item())


In [ ]:
# Course completion: Restore sampling statistics and perplexity comparisons.
highest_perp = eos_df.nlargest(1, "perplexity")
print(highest_perp["text"].item())


In [ ]:
dist_df.to_csv(RESULTS_DIR / "dist_df.csv", index=True)
eos_df.to_csv(RESULTS_DIR / "eos_df.csv", index=True)
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
